# Риманова модель с гейтом по людям на многообразии

1. **Признаки** (`bci/riemann.py`). Ковариации окон в полосе 4–40 Гц. Сессия перецентрируется на лету: опорная точка — бегущее среднее её ковариаций, метки не нужны.
   - `x` — окно в касательном пространстве относительно сессии. Индивидуальность человека из него убрана.
   - `fp` — где опорная точка сессии лежит на многообразии, то есть «отпечаток» человека.
2. **Люди на многообразии.** У каждого train-участника есть центр (средний отпечаток) и обучаемый эмбеддинг.
3. **Гейт.** Эмбеддинг сессии — смесь эмбеддингов ближайших людей с весами `softmax(-d² / τ)`; температура τ тоже обучается. Эмбеддинг через FiLM подстраивает классификатор.
   - Знакомый человек (его поздние сессии в sealed-тесте): гейт находит его самого.
   - Новый человек: гейт берёт похожих на него.
   - Чтобы модель умела работать через соседей, при обучении собственный центр человека иногда прячем от гейта.

Оценка везде как на платформе: окна идут по одному, каждая сессия начинается с чистой копии модели.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from bci.data import load_split, concat
from bci.riemann import RiemannFeatures, StreamingDecoder
from bci.evaluation import (score, evaluate, compare_with_baselines,
                            cross_validate, leave_one_subject_out, later_sessions)

torch.manual_seed(0); np.random.seed(0)
DATA_DIR = "../data"
train, val, test = (load_split(s, DATA_DIR) for s in ["train", "val", "test"])
dev = concat(train, val)  # для кросс-валидации; test не трогаем до самого конца
T = lambda a: torch.as_tensor(a, dtype=torch.float32)

## Люди на многообразии
Каждая точка — сессия (её опорная точка в конце записи), цвет — участник. Сессии одного человека лежат рядом, на этом и держится гейт.

In [ ]:
features = RiemannFeatures()
x_dev, fp_dev = features.fit_transform(dev)
last = [idx[-1] for idx in dev.streams()]  # конец сессии = вся сессия в опорной точке

z = PCA(2).fit_transform(fp_dev[last])
plt.scatter(*z.T, c=pd.factorize(dev.subject[last])[0], cmap="tab10", s=60)
for (a, b), s in zip(z, dev.subject[last]):
    plt.annotate(s.split("/")[-1], (a, b), fontsize=8)
plt.title("сессии на многообразии (PCA лог-евклидовых координат)");

## Ориентир: перецентрирование + логистическая регрессия

In [ ]:
def fit_logreg(train):
    features = RiemannFeatures()
    x, _ = features.fit_transform(train)
    clf = LogisticRegression(C=0.01, max_iter=2000, class_weight="balanced").fit(x, train.y)
    return StreamingDecoder(features, lambda x, fp: int(clf.predict(x[None])[0]))

## Модель: гейт по многообразию + обучаемые эмбеддинги людей

In [ ]:
class ManifoldGatedRiemann(nn.Module):
    def __init__(self, n_feat, centers, tau, n_classes=3, dim=32, emb=8):
        super().__init__()
        self.register_buffer("centers", T(centers))        # где лежат train-участники
        self.table = nn.Embedding(len(centers), emb)        # обучаемый эмбеддинг каждого
        self.log_tau = nn.Parameter(T(np.log(tau)))         # температура гейта, учится
        self.proj, self.drop = nn.Linear(n_feat, dim), nn.Dropout(0.3)
        self.film = nn.Linear(emb, 2 * dim)
        self.head = nn.Linear(dim, n_classes)

    def gate(self, fp, hide=None):
        """Веса train-участников для сессии с отпечатком fp; hide прячет центры."""
        logits = -torch.cdist(fp, self.centers) ** 2 / self.log_tau.exp()
        if hide is not None:
            logits = logits.masked_fill(hide, float("-inf"))
        return torch.softmax(logits, dim=-1)

    def forward(self, x, fp, hide=None):
        e = self.gate(fp, hide) @ self.table.weight
        gamma, beta = self.film(e).chunk(2, dim=-1)
        return self.head(torch.relu(self.proj(self.drop(x)) * (1 + gamma) + beta))

In [ ]:
def fit_gated(train, epochs=150, hide_own=0.5, val=None):
    features = RiemannFeatures()
    x, fp = features.fit_transform(train)
    subjects = sorted(set(train.subject))
    s = np.array([subjects.index(v) for v in train.subject])
    centers = np.stack([fp[s == k].mean(0) for k in range(len(subjects))])
    d2 = ((centers[:, None] - centers[None]) ** 2).sum(-1)

    net = ManifoldGatedRiemann(x.shape[1], centers, tau=np.median(d2[d2 > 0]))
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    weights = T(len(s) / (3 * np.bincount(train.y, minlength=3)))
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
    X, F, Y = T(x), T(fp), torch.as_tensor(train.y)
    own = torch.nn.functional.one_hot(torch.as_tensor(s), len(subjects)).bool()
    if val is not None:
        x_va, fp_va = features.transform(val)

    history = []
    for _ in range(epochs):
        net.train(); opt.zero_grad()
        hide = own & (torch.rand(len(s), 1) < hide_own)  # иногда прячем самого себя
        loss_fn(net(X, F, hide), Y).backward(); opt.step()
        if val is not None:
            net.eval()
            with torch.no_grad():
                history.append(score(val, net(T(x_va), T(fp_va)).argmax(-1).numpy())["session_mean"])

    net.eval()
    def classify(x, fp):
        with torch.no_grad():
            return int(net(T(x[None]), T(fp[None])).argmax())
    model = StreamingDecoder(features, classify)
    model.net, model.subjects = net, subjects
    return (model, history) if val is not None else model

Проверка числа эпох: кривая val (новые люди) не должна заметно падать к пунктиру. Если падает, уменьшите `EPOCHS`.

In [ ]:
EPOCHS = 150
model, history = fit_gated(train, epochs=300, val=val)
print(f"τ после обучения = {model.net.log_tau.exp().item():.1f}")
ax = pd.Series(history).rolling(25, min_periods=1).mean().plot(title="val session_mean (сглажено)")
ax.axvline(EPOCHS, ls="--", c="gray");

## Кросс-валидация на train + val
- **Новые люди**: leave-one-subject-out, каждый человек по очереди выносится в тест.
- **Поздние сессии знакомых людей**: последняя сессия каждого — тест, ранние — train.

Вместе они покрывают обе половины sealed-теста.

In [ ]:
models = {"Riemann + LogReg": fit_logreg,
          "Riemann + manifold gate": lambda tr: fit_gated(tr, epochs=EPOCHS)}

loso = {name: cross_validate(fit, leave_one_subject_out(dev)) for name, fit in models.items()}
per_subject = pd.DataFrame({name: r["session_mean"] for name, r in loso.items()})
per_subject.loc["mean"] = per_subject.mean()
per_subject.round(3)

In [ ]:
later = {name: cross_validate(fit, later_sessions(dev)) for name, fit in models.items()}
pd.DataFrame({
    "новые люди": {n: r["session_mean"].mean() for n, r in loso.items()},
    "новые люди (std)": {n: r["session_mean"].std() for n, r in loso.items()},
    "поздние сессии": {n: r["session_mean"].iloc[0] for n, r in later.items()},
}).round(3)

Узнаёт ли гейт знакомого человека по поздней сессии? Доля окон, где самый большой вес у него самого (случайный уровень = 1 / число людей).

In [ ]:
_, dev_early, dev_late = next(later_sessions(dev))
gated = fit_gated(dev_early, epochs=EPOCHS)
_, fp_late = gated.features.transform(dev_late)
with torch.no_grad():
    picked = gated.net.gate(T(fp_late)).argmax(-1).numpy()
hit = np.array(gated.subjects)[picked] == dev_late.subject
print(f"гейт выбрал самого человека в {hit.mean():.0%} окон (случайно: {1 / len(gated.subjects):.0%})")

## Официальный test: сравнение с опубликованными бейзлайнами
Обучаем на train (как бейзлайны) и один раз оцениваем на test.

In [ ]:
results = {name: evaluate(fit(train), test) for name, fit in models.items()}
compare_with_baselines(results)